# Симуляция обработки очереди сообщений с SimPy

Этот notebook моделирует discrete-event simulation для очереди из **200 000 сообщений** и **20 потоков/воркеров**.

Каждый воркер берет одно сообщение, обрабатывает его, затем берет следующее. Время обработки одного сообщения задается распределением с параметрами:

- 50-й перцентиль: **1.7 секунды**
- 95-й перцентиль: **3.6 секунды**

В результате считаются:

- сколько сообщений обрабатывается всеми потоками в секунду и минуту;
- сколько сообщений обрабатывается каждым потоком в секунду и минуту;
- сколько времени уходит на обработку всех сообщений;
- графики скорости обработки во времени.

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import simpy

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

## Параметры симуляции

Для времени обработки используется логнормальное распределение. Оно удобно для таких задач, потому что время всегда положительное, а длинный правый хвост естественно описывает редкие медленные обработки.

Параметры распределения подбираются так, чтобы совпали заданные перцентили:

- `median = exp(mu) = 1.7`
- `p95 = exp(mu + sigma * z95) = 3.6`, где `z95 = 1.64485`

In [ ]:
@dataclass(frozen=True)
class SimulationConfig:
    messages: int = 200_000
    workers: int = 20
    median_seconds: float = 1.7
    p95_seconds: float = 3.6
    random_seed: int = 42


config = SimulationConfig()

Z95 = 1.6448536269514722
lognormal_mu = math.log(config.median_seconds)
lognormal_sigma = math.log(config.p95_seconds / config.median_seconds) / Z95

print(f"mu: {lognormal_mu:.4f}")
print(f"sigma: {lognormal_sigma:.4f}")
print(f"theoretical median: {math.exp(lognormal_mu):.3f} sec")
print(f"theoretical p95: {math.exp(lognormal_mu + lognormal_sigma * Z95):.3f} sec")
print(f"theoretical mean: {math.exp(lognormal_mu + 0.5 * lognormal_sigma**2):.3f} sec")

## SimPy-модель

Модель использует `simpy.Store` как очередь сообщений. Воркер ждет сообщение, фиксирует время старта, выполняет `env.timeout(processing_time)`, затем фиксирует завершение.

In [ ]:
def sample_processing_time(rng: np.random.Generator) -> float:
    return float(rng.lognormal(mean=lognormal_mu, sigma=lognormal_sigma))


def worker(env: simpy.Environment, worker_id: int, queue: simpy.Store, rng: np.random.Generator, records: list[dict]):
    while True:
        message_id = yield queue.get()
        if message_id is None:
            break

        start_time = env.now
        duration = sample_processing_time(rng)
        yield env.timeout(duration)
        finish_time = env.now

        records.append(
            {
                "message_id": message_id,
                "worker_id": worker_id,
                "start_time": start_time,
                "finish_time": finish_time,
                "duration": duration,
            }
        )


def run_simulation(config: SimulationConfig) -> pd.DataFrame:
    env = simpy.Environment()
    queue = simpy.Store(env)
    rng = np.random.default_rng(config.random_seed)
    records: list[dict] = []

    # Для 200k сообщений список в памяти небольшой, а Store делает модель явной и читаемой.
    queue.items = list(range(config.messages)) + [None] * config.workers

    for worker_id in range(config.workers):
        env.process(worker(env, worker_id, queue, rng, records))

    env.run()

    result = pd.DataFrame.from_records(records)
    return result.sort_values("finish_time", ignore_index=True)


results = run_simulation(config)
results.head()

## Проверка распределения времени обработки

In [ ]:
duration_stats = results["duration"].describe(percentiles=[0.5, 0.95, 0.99]).to_frame("seconds")
duration_stats

## Итоговая скорость и полное время обработки

In [ ]:
total_messages = len(results)
total_time_seconds = results["finish_time"].max()
total_time_minutes = total_time_seconds / 60
total_time_hours = total_time_seconds / 3600

overall_messages_per_second = total_messages / total_time_seconds
overall_messages_per_minute = overall_messages_per_second * 60

summary = pd.DataFrame(
    [
        {"metric": "messages", "value": total_messages},
        {"metric": "total_time_seconds", "value": total_time_seconds},
        {"metric": "total_time_minutes", "value": total_time_minutes},
        {"metric": "total_time_hours", "value": total_time_hours},
        {"metric": "all_workers_messages_per_second", "value": overall_messages_per_second},
        {"metric": "all_workers_messages_per_minute", "value": overall_messages_per_minute},
    ]
)

summary

## Скорость каждого потока

`messages_per_second_total_window` считает среднюю скорость воркера на всем интервале симуляции. Это удобно для сравнения с общей скоростью системы.

`messages_per_second_active_window` считает скорость только за активное время конкретного воркера, от первого старта до последнего завершения.

In [ ]:
worker_summary = (
    results.groupby("worker_id")
    .agg(
        messages=("message_id", "count"),
        first_start=("start_time", "min"),
        last_finish=("finish_time", "max"),
        mean_duration=("duration", "mean"),
        median_duration=("duration", "median"),
        p95_duration=("duration", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)

worker_summary["active_seconds"] = worker_summary["last_finish"] - worker_summary["first_start"]
worker_summary["messages_per_second_total_window"] = worker_summary["messages"] / total_time_seconds
worker_summary["messages_per_minute_total_window"] = worker_summary["messages_per_second_total_window"] * 60
worker_summary["messages_per_second_active_window"] = worker_summary["messages"] / worker_summary["active_seconds"]
worker_summary["messages_per_minute_active_window"] = worker_summary["messages_per_second_active_window"] * 60

worker_summary

## График общей скорости обработки

Ниже скорость считается по завершенным сообщениям в окнах по 60 секунд.

In [ ]:
def build_throughput(results: pd.DataFrame, bin_seconds: int = 60) -> pd.DataFrame:
    max_time = math.ceil(results["finish_time"].max())
    bins = np.arange(0, max_time + bin_seconds, bin_seconds)

    throughput = (
        results.assign(time_bin=pd.cut(results["finish_time"], bins=bins, right=False, include_lowest=True))
        .groupby("time_bin", observed=False)
        .size()
        .rename("messages")
        .reset_index()
    )

    throughput["window_start"] = throughput["time_bin"].apply(lambda interval: interval.left).astype(float)
    throughput["window_minute"] = throughput["window_start"] / 60
    throughput["messages_per_second"] = throughput["messages"] / bin_seconds
    throughput["messages_per_minute"] = throughput["messages_per_second"] * 60
    return throughput.drop(columns="time_bin")


throughput_60s = build_throughput(results, bin_seconds=60)
throughput_60s.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=throughput_60s, x="window_minute", y="messages_per_second", ax=ax)
ax.axhline(overall_messages_per_second, color="crimson", linestyle="--", label="Средняя скорость")
ax.set_title("Общая скорость обработки: сообщений в секунду")
ax.set_xlabel("Минута симуляции")
ax.set_ylabel("Сообщений / сек")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=throughput_60s, x="window_minute", y="messages_per_minute", ax=ax)
ax.axhline(overall_messages_per_minute, color="crimson", linestyle="--", label="Средняя скорость")
ax.set_title("Общая скорость обработки: сообщений в минуту")
ax.set_xlabel("Минута симуляции")
ax.set_ylabel("Сообщений / мин")
ax.legend()
plt.show()

## График скорости по потокам

Каждая линия — отдельный поток. Окно агрегации также 60 секунд.

In [ ]:
def build_worker_throughput(results: pd.DataFrame, bin_seconds: int = 60) -> pd.DataFrame:
    max_time = math.ceil(results["finish_time"].max())
    bins = np.arange(0, max_time + bin_seconds, bin_seconds)

    worker_throughput = (
        results.assign(time_bin=pd.cut(results["finish_time"], bins=bins, right=False, include_lowest=True))
        .groupby(["worker_id", "time_bin"], observed=False)
        .size()
        .rename("messages")
        .reset_index()
    )

    worker_throughput["window_start"] = worker_throughput["time_bin"].apply(lambda interval: interval.left).astype(float)
    worker_throughput["window_minute"] = worker_throughput["window_start"] / 60
    worker_throughput["messages_per_second"] = worker_throughput["messages"] / bin_seconds
    worker_throughput["messages_per_minute"] = worker_throughput["messages_per_second"] * 60
    return worker_throughput.drop(columns="time_bin")


worker_throughput_60s = build_worker_throughput(results, bin_seconds=60)
worker_throughput_60s.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=worker_throughput_60s,
    x="window_minute",
    y="messages_per_minute",
    hue="worker_id",
    palette="tab20",
    linewidth=1,
    alpha=0.8,
    legend=False,
    ax=ax,
)
ax.set_title("Скорость обработки по потокам")
ax.set_xlabel("Минута симуляции")
ax.set_ylabel("Сообщений / мин на поток")
plt.show()

## Распределение скорости потоков

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=worker_summary,
    x="worker_id",
    y="messages_per_minute_total_window",
    color="#4C78A8",
    ax=axes[0],
)
axes[0].set_title("Средняя скорость по потокам")
axes[0].set_xlabel("Поток")
axes[0].set_ylabel("Сообщений / мин")

sns.histplot(results["duration"], bins=80, color="#F58518", ax=axes[1])
axes[1].axvline(config.median_seconds, color="black", linestyle="--", label="Заданная медиана")
axes[1].axvline(config.p95_seconds, color="crimson", linestyle="--", label="Заданный p95")
axes[1].set_title("Распределение времени обработки")
axes[1].set_xlabel("Секунды")
axes[1].set_ylabel("Количество сообщений")
axes[1].legend()

plt.tight_layout()
plt.show()

## Короткий текстовый вывод

In [ ]:
print(f"Обработано сообщений: {total_messages:,}")
print(f"Полное время обработки: {total_time_seconds:,.1f} сек = {total_time_minutes:,.1f} мин = {total_time_hours:,.2f} ч")
print(f"Все потоки: {overall_messages_per_second:.2f} сообщений/сек = {overall_messages_per_minute:.1f} сообщений/мин")
print(
    "Один поток в среднем: "
    f"{worker_summary['messages_per_second_total_window'].mean():.3f} сообщений/сек = "
    f"{worker_summary['messages_per_minute_total_window'].mean():.1f} сообщений/мин"
)